# Lab 4: Adaptive Optimisers, Regularisation and Optuna

**DATA425 | Foundations of Deep Learning**

## What this lab is about

So far, we have looked at how gradients are computed and how they update weights. This lab focuses on making training work well in practice.

We will use a small handwritten digits dataset to explore three practical questions:

1. **Which optimiser should we use?** Different optimisers use gradient information differently.
2. **How do we diagnose overfitting?** Training curves can reveal when a model is memorising rather than generalising.
3. **How do we choose hyperparameters?** Tools such as Optuna help search over choices like learning rate, dropout, and layer size.

The dataset is small, so the notebook runs quickly. The same workflow scales to larger datasets: split the data, compare models fairly, use validation data for choices, and keep test data for the final check.


## 0. Setup

Run the setup cells first. The Optuna search is intentionally small so the lab stays interactive.

You can increase `OPTUNA_TRIALS`, `OPTUNA_EPOCHS`, and `FINAL_EPOCHS` later if you want a more thorough search. The trade-off is runtime.


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED = {
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "tensorflow": "tensorflow",
    "optuna": "optuna"
}

missing = [package for module, package in REQUIRED.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print("Installing missing packages:", ", ".join(missing))
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Automatic installation failed. Install these packages manually or run this notebook "
            "in an environment with internet access: " + ", ".join(missing)
        ) from exc
else:
    print("All required packages are already installed.")

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

try:
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.config.threading.set_intra_op_parallelism_threads(1)
except RuntimeError:
    pass

SEED = 425
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

OPTUNA_TRIALS = 3
OPTUNA_EPOCHS = 2
FINAL_EPOCHS = 5

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

The setup cell defines small runtime constants for the search. These values are chosen for teaching speed, not because they are the best possible values for every problem.


## 1. Adaptive optimisers on a narrow loss surface

A single learning rate can be awkward when the loss surface has different shapes in different directions. One direction may be steep, while another direction may be flat.

- **SGD** uses the current gradient directly.
- **RMSProp** rescales updates using recent squared gradients.
- **Adam** combines momentum with adaptive rescaling.

This toy loss surface lets us see optimiser behaviour visually before we use the same optimisers in a neural network.


In [ ]:
def surface_loss(point):
    x, y = point
    return 0.01 * x**2 + y**2

def surface_grad(point):
    x, y = point
    return np.array([0.02 * x, 2.0 * y])

def run_optimizer(name, steps=70, lr=0.25):
    p = np.array([8.0, 5.0], dtype=float)
    path = [p.copy()]
    v = np.zeros_like(p)
    s = np.zeros_like(p)
    m = np.zeros_like(p)
    beta1, beta2 = 0.9, 0.999
    for t in range(1, steps + 1):
        g = surface_grad(p)
        if name == "sgd":
            update = g
        elif name == "rmsprop":
            s = 0.9 * s + 0.1 * (g ** 2)
            update = g / (np.sqrt(s) + 1e-8)
        elif name == "adam":
            m = beta1 * m + (1 - beta1) * g
            s = beta2 * s + (1 - beta2) * (g ** 2)
            m_hat = m / (1 - beta1 ** t)
            s_hat = s / (1 - beta2 ** t)
            update = m_hat / (np.sqrt(s_hat) + 1e-8)
        p = p - lr * update
        path.append(p.copy())
    return np.array(path)

paths = {
    "SGD": run_optimizer("sgd", lr=0.35),
    "RMSProp": run_optimizer("rmsprop", lr=0.20),
    "Adam": run_optimizer("adam", lr=0.22),
}

xs = np.linspace(-9, 9, 250)
ys = np.linspace(-6, 6, 250)
xx, yy = np.meshgrid(xs, ys)
zz = 0.01 * xx**2 + yy**2

plt.contour(xx, yy, zz, levels=30)
for name, path in paths.items():
    plt.plot(path[:, 0], path[:, 1], marker="o", markersize=2, label=name)
plt.xlabel("parameter 1")
plt.ylabel("parameter 2")
plt.title("Adaptive optimisers can move differently in each direction")
plt.legend()
plt.show()

Focus on the shape of the paths. Adaptive optimisers often move more comfortably through narrow or uneven loss landscapes because they adjust the update scale separately for different parameters.


## 2. Load the digits dataset

We use a built-in dataset of handwritten digits from 0 to 9. Each image is 8 by 8 pixels, so it is much smaller than a typical real image, but it is still useful for practising the full workflow.

We create three splits:

- **training data**, used to fit weights;
- **validation data**, used to compare choices during development;
- **test data**, kept for the final evaluation only.

Keeping these roles separate is one of the most important habits in machine learning. If the test set influences model choices, it stops being an honest final check.


In [ ]:
digits = load_digits()
X_all = digits.images.astype("float32") / 16.0
y_all = digits.target.astype("int64")

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_all, y_all, test_size=0.20, random_state=SEED, stratify=y_all
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=SEED, stratify=y_train_full
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

fig, axes = plt.subplots(2, 5, figsize=(8, 4))
for ax, image, label in zip(axes.ravel(), X_train[:10], y_train[:10]):
    ax.imshow(image, cmap="gray")
    ax.set_title(f"label {label}")
    ax.axis("off")
plt.tight_layout()
plt.show()

The images are scaled to values between 0 and 1. Scaling inputs usually makes neural network training more stable because the optimiser does not have to handle unnecessarily large feature values.


## 3. Compare optimisers in Keras

All models in this section use the same architecture. The only thing we change is the optimiser.

This is important for a fair comparison. If we changed the optimiser and the architecture at the same time, we would not know which change caused the difference.

The model is a small dense neural network:

1. Flatten the 8 by 8 image into a vector.
2. Pass it through hidden layers with ReLU activations.
3. Use a 10-unit softmax output for the digit classes.


In [ ]:
def build_digits_model(dropout=0.0, l2_strength=0.0):
    reg = regularizers.l2(l2_strength) if l2_strength > 0 else None
    model = keras.Sequential([
        keras.Input(shape=(8, 8)),
        layers.Flatten(),
        layers.Dense(128, activation="relu", kernel_regularizer=reg),
        layers.Dropout(dropout),
        layers.Dense(64, activation="relu", kernel_regularizer=reg),
        layers.Dropout(dropout),
        layers.Dense(10, activation="softmax"),
    ])
    return model


def plot_histories(histories, metric="val_accuracy", title="Validation accuracy"):
    for name, history in histories.items():
        plt.plot(history.history[metric], label=name)
    plt.xlabel("epoch")
    plt.ylabel(metric)
    plt.title(title)
    plt.legend()
    plt.show()

optimizers = {
    "SGD": keras.optimizers.SGD(learning_rate=0.08, momentum=0.9),
    "RMSProp": keras.optimizers.RMSprop(learning_rate=0.001),
    "Adam": keras.optimizers.Adam(learning_rate=0.001),
}

histories = {}
for name, optimizer in optimizers.items():
    keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    model = build_digits_model()
    model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    histories[name] = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                                epochs=5, batch_size=64, verbose=0)
    print(f"{name:7s} final validation accuracy: {histories[name].history['val_accuracy'][-1]:.3f}")

plot_histories(histories, "val_accuracy", "Optimiser comparison")

The validation accuracy curves show how quickly each optimiser learns on this task. Do not treat the result as a universal ranking. Optimiser performance depends on the model, data, learning rate, and training time.


## 4. Diagnose overfitting

Overfitting happens when a model learns the training data too specifically and does not generalise well to new examples.

A common pattern is:

- training accuracy keeps improving;
- validation accuracy stalls or gets worse;
- the gap between training and validation performance grows.

To make this pattern easy to see, we train a large model on a deliberately small subset of the training data.


In [ ]:
small_X = X_train[:160]
small_y = y_train[:160]

keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

big_model = keras.Sequential([
    keras.Input(shape=(8, 8)),
    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(256, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax"),
])
big_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])

history_big = big_model.fit(small_X, small_y, validation_data=(X_val, y_val),
                            epochs=15, batch_size=32, verbose=0)

plt.plot(history_big.history["accuracy"], label="training accuracy")
plt.plot(history_big.history["val_accuracy"], label="validation accuracy")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("A large model can overfit a small training set")
plt.legend()
plt.show()

If the training curve improves while the validation curve stops improving, that is a warning sign. The model may be memorising details of the small training subset.


## 5. Add regularisation

Regularisation adds constraints or noise that make memorisation harder. Here we use three common tools together:

- **L2 regularisation**, which discourages very large weights;
- **dropout**, which randomly switches off hidden units during training;
- **early stopping**, which restores weights from the best validation epoch.

These methods do not make a model simpler in exactly the same way, but they share a goal: improve generalisation by reducing over-reliance on the training set.


In [ ]:
keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

regularised_model = build_digits_model(dropout=0.25, l2_strength=1e-3)
regularised_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                          loss="sparse_categorical_crossentropy", metrics=["accuracy"])

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)
history_regularised = regularised_model.fit(
    small_X, small_y,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=0,
)

plt.plot(history_big.history["val_accuracy"], label="large model")
plt.plot(history_regularised.history["val_accuracy"], label="regularised model")
plt.xlabel("epoch")
plt.ylabel("validation accuracy")
plt.title("Regularisation can improve validation behaviour")
plt.legend()
plt.show()

Regularisation is not guaranteed to improve every model, but it often improves validation behaviour when the unregularised model is clearly overfitting.


## 6. Hyperparameter tuning with Optuna

Hyperparameters are choices we make before training, such as learning rate, number of hidden layers, number of units, dropout rate, and optimiser type.

Optuna automates part of the search process:

1. Suggest a set of hyperparameters.
2. Train a model using those hyperparameters.
3. Measure validation performance.
4. Use the results to guide future suggestions.

The search in this lab is small so it runs quickly. In a real project, you would usually run more trials and keep careful records of the search space.


In [ ]:
def objective(trial):
    keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED + trial.number)

    learning_rate = trial.suggest_float("learning_rate", 1e-4, 3e-2, log=True)
    units = trial.suggest_categorical("units", [32, 64, 128])
    n_layers = trial.suggest_int("n_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.0, 0.35)
    optimizer_name = trial.suggest_categorical("optimizer", ["adam", "rmsprop", "sgd"])

    model = keras.Sequential([keras.Input(shape=(8, 8)), layers.Flatten()])
    for _ in range(n_layers):
        model.add(layers.Dense(units, activation="relu"))
        if dropout > 0:
            model.add(layers.Dropout(dropout))
    model.add(layers.Dense(10, activation="softmax"))

    if optimizer_name == "adam":
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == "rmsprop":
        optimizer = keras.optimizers.RMSprop(learning_rate=learning_rate)
    else:
        optimizer = keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)

    model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                        epochs=OPTUNA_EPOCHS, batch_size=64, verbose=0)
    return max(history.history["val_accuracy"])

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=OPTUNA_TRIALS)

print("Best validation accuracy:", f"{study.best_value:.3f}")
print("Best hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

The best trial is the best within the small search we allowed. It is not proof that these are the best possible hyperparameters. Hyperparameter tuning is always limited by the search space, runtime, and validation split.


## 7. Train the best Optuna model

After choosing hyperparameters using the validation set, we rebuild the model and train it using the available development data. Then we evaluate once on the held-out test set.

This sequence matters:

1. Use training data to fit weights.
2. Use validation data to choose hyperparameters.
3. Use test data only at the end.

The test accuracy is the best estimate in this notebook of how the selected approach performs on unseen data from the same dataset.


In [ ]:
best = study.best_params
keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

best_model = keras.Sequential([keras.Input(shape=(8, 8)), layers.Flatten()])
for _ in range(best["n_layers"]):
    best_model.add(layers.Dense(best["units"], activation="relu"))
    if best["dropout"] > 0:
        best_model.add(layers.Dropout(best["dropout"]))
best_model.add(layers.Dense(10, activation="softmax"))

if best["optimizer"] == "adam":
    best_optimizer = keras.optimizers.Adam(learning_rate=best["learning_rate"])
elif best["optimizer"] == "rmsprop":
    best_optimizer = keras.optimizers.RMSprop(learning_rate=best["learning_rate"])
else:
    best_optimizer = keras.optimizers.SGD(learning_rate=best["learning_rate"], momentum=0.9)

best_model.compile(optimizer=best_optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history_best = best_model.fit(X_train_full, y_train_full,
                              epochs=FINAL_EPOCHS, batch_size=64, verbose=0)

test_probs = best_model.predict(X_test, verbose=0)
test_preds = np.argmax(test_probs, axis=1)
print("Test accuracy:", f"{accuracy_score(y_test, test_preds):.3f}")
print("Confusion matrix:")
print(confusion_matrix(y_test, test_preds))

Read the confusion matrix by row. It shows which true classes are being confused with which predicted classes. This is often more informative than accuracy alone.


## Your turn

Try one small change at a time so you can interpret the result.

Suggested experiments:

1. Increase `OPTUNA_TRIALS` to 20 and compare the best validation accuracy.
2. Add `l2_strength` to the Optuna search space.
3. Add `batch_size` to the Optuna search space.
4. Try a smaller or larger hidden layer size.

Keep the test set for the final evaluation. Use the validation set for exploration.


In [ ]:
# Optional exercise scaffold.
# Replace pass with your own Optuna objective if you want to extend the search.
pass

## Summary

You have now practised several practical training tools:

- SGD, RMSProp, and Adam use gradient information in different ways.
- Validation curves help diagnose underfitting and overfitting.
- L2 regularisation, dropout, and early stopping can reduce overfitting.
- Hyperparameter tuning searches over modelling choices that are not learned directly by gradient descent.
- The test set should be used only for the final check after model choices have been made.

The main takeaway is that training a neural network is an experimental process. Good experiments change one idea at a time and keep validation and test data roles separate.
